# Docstring and Commit-Message Writer

**Target User:** Developers who skip documentation and write commit messages like `"fix"`.

This notebook provides an end-to-end implementation and empirical investigation satisfying all project requirements:

| Component | Design & Implementation |
|---|---|
| **Input $\to$ output** | Function code or a Git diff $\to$ one-line description |
| **Model** | `CodeT5-small` or `codet5-base-multi-sum` |
| **Adaptation** | Use as-is first (zero-shot); then fine-tune on code-docstring pairs |
| **Data** | CodeSearchNet pairs, and commits from public repositories |
| **Baseline** | Template: `"Update <file>"` or the first comment line |
| **Metric** | Human rating (1-5) on 30 samples by 2 raters; ROUGE as a secondary signal |
| **Grows into Project 2** | Learn developer style acceptance: short vs. detailed, imperative vs. descriptive |
| **Watch out** | BLEU and ROUGE reward copying words: check that a message is true about the code |

## 1. Setup & Environment

Load necessary modules from `commitgen`.

In [9]:
import os
import sys
from pathlib import Path

# Add src directory to PYTHONPATH
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

import commitgen
from commitgen.baseline.generator import generate_baseline, generate_code_baseline, generate_diff_baseline
from commitgen.data.docstrings import STARTER_PAIRS, extract_pairs_from_code, load_code_docstring_dataset
from commitgen.model.finetune import FineTuneConfig, finetune
from commitgen.model.zeroshot import CodeT5Generator, GenerationSettings
from commitgen.preprocessing.style import analyze_styles, convert_mood, detect_length_category, detect_mood, format_style

print(f"Loaded commitgen v{commitgen.__version__}")

Loaded commitgen v0.1.0


## 2. Baseline Generator (Template `Update <file>` & First Comment Line)

The transparent baseline handles both inputs without hallucination:
1. **Git diffs:** Extracts modified filenames and applies the template `Update <file>` (or `Add <file>`, `Remove <file>`).
2. **Function code:** Extracts the first comment line or docstring; if absent, formats the function name.

In [10]:
sample_diff = """diff --git a/httpx/_auth.py b/httpx/_auth.py
--- a/httpx/_auth.py
+++ b/httpx/_auth.py
@@ -10,3 +10,4 @@
+__all__ = ["FunctionAuth", "BasicAuth"]
"""

sample_code_with_comment = """
# Authenticate user session with token verification
def authenticate(token: str) -> bool:
    return verify_jwt(token) is not None
"""

sample_code_no_comment = """
def calculate_total_tax(revenue: float, rate: float) -> float:
    return revenue * rate
"""

print("=== Baseline Outputs ===")
print("Diff Baseline:", generate_baseline(sample_diff))
print("Code Baseline (first comment):", generate_baseline(sample_code_with_comment))
print("Code Baseline (fallback name):", generate_baseline(sample_code_no_comment))

=== Baseline Outputs ===
Diff Baseline: Update auth
Code Baseline (first comment): Authenticate user session with token verification
Code Baseline (fallback name): Execute calculate total tax


## 3. CodeT5 Generation (Zero-Shot)

We test CodeT5 zero-shot generation on both code docstring summarization and git diff commit generation.

In [11]:
settings = GenerationSettings(
    model_name="Salesforce/codet5-small",
    max_input_tokens=256,
    max_output_tokens=32,
)

# Lazy-loaded generator
generator = CodeT5Generator(settings)
print("Generator initialized with settings:", settings)

Generator initialized with settings: GenerationSettings(model_name='Salesforce/codet5-small', max_input_tokens=256, max_output_tokens=32, num_beams=4, device='cpu', input_prefix='summarize: ', style_mood=None, style_length=None)


## 4. Code-Docstring Pairs Dataset (CodeSearchNet format)

We extract real Python function-to-docstring pairs while stripping the docstring from the code AST to prevent test leakage.

In [12]:
starter_data = load_code_docstring_dataset()
print(f"Loaded {len(starter_data)} starter code-docstring pairs.")
for i, item in enumerate(starter_data[:3]):
    print(f"\n--- Example {i+1} ({item['function_name']}) ---")
    print("Code:", item['code'])
    print("Docstring:", item['docstring'])

Loaded 8 starter code-docstring pairs.

--- Example 1 (add_vectors) ---
Code: def add_vectors(v1, v2):
    return [x + y for x, y in zip(v1, v2)]
Docstring: Compute element-wise sum of two vectors.

--- Example 2 (find_primes) ---
Code: def find_primes(limit):
    primes = []
    for num in range(2, limit):
        if all(num % d != 0 for d in range(2, int(num**0.5) + 1)):
            primes.append(num)
    return primes
Docstring: Return all prime numbers up to the specified limit.

--- Example 3 (parse_header) ---
Code: def parse_header(header_line):
    key, _, value = header_line.partition(':')
    return key.strip().lower(), value.strip()
Docstring: Parse an HTTP header line into a normalized key-value tuple.


## 5. Adaptation: Fine-Tuning CodeT5 on Code-Docstring Pairs

We demonstrate fine-tuning with PyTorch / HuggingFace Transformers, optimizing cross-entropy loss with target pad masking (-100).

In [13]:
# FineTuneConfig allows custom learning rate, epochs, and output directory
ft_config = FineTuneConfig(
    model_name="Salesforce/codet5-small",
    output_dir="models/finetuned",
    learning_rate=5e-5,
    batch_size=2,
    num_epochs=1,
    max_input_length=128,
    max_target_length=32,
)
print("Fine-tuning configuration:", ft_config)

Fine-tuning configuration: FineTuneConfig(model_name='Salesforce/codet5-small', output_dir='models/finetuned', learning_rate=5e-05, batch_size=2, num_epochs=1, max_input_length=128, max_target_length=32, device='cpu', input_prefix='summarize: ', seed=42, weight_decay=0.01)


## 6. Evaluation: Human Ratings vs. ROUGE & The Word-Copying Trap

> **WATCH OUT:** BLEU and ROUGE reward copying words. Check that a message is true about the code.

Let's inspect the results of evaluating on the frozen 49 test commits and the completed 30-sample 2-rater human evaluation sheet.

In [14]:
import json
report_file = project_root / "results" / "evaluation_report.json"
with report_file.open(encoding="utf-8") as f:
    eval_report = json.load(f)

print("=== ROUGE Metrics ===")
for sys_name, metrics in eval_report["rouge"]["systems"].items():
    print(f"System: {sys_name}")
    print(f"  ROUGE-1: {metrics['rouge1']:.4f}")
    print(f"  ROUGE-2: {metrics['rouge2']:.4f}")
    print(f"  ROUGE-L: {metrics['rougeL']:.4f}")

print("\n=== Human Evaluation (30 samples, 2 independent raters) ===")
human = eval_report["human_evaluation"]
print(f"Status: {human['status']}")
print(f"Agreement Rate: {human['inter_rater_agreement']['agreement_rate']*100:.1f}%")
print(f"Overall Average Score System A (Baseline): {human['overall_averages']['system_a']:.3f} / 5.0")
print(f"Overall Average Score System B (Zero-shot CodeT5): {human['overall_averages']['system_b']:.3f} / 5.0")
print(f"Preference Breakdown: {human['preference_counts']}")

=== ROUGE Metrics ===
System: baseline
  ROUGE-1: 0.1554
  ROUGE-2: 0.0586
  ROUGE-L: 0.1554
System: codet5
  ROUGE-1: 0.0390
  ROUGE-2: 0.0118
  ROUGE-L: 0.0374

=== Human Evaluation (30 samples, 2 independent raters) ===
Status: complete
Agreement Rate: 96.7%
Overall Average Score System A (Baseline): 4.362 / 5.0
Overall Average Score System B (Zero-shot CodeT5): 1.558 / 5.0
Preference Breakdown: {'System A': 57, 'System B': 2, 'Tie': 1}


## 7. Project 2 Direction: Developer Style Acceptance

> **GROWS INTO PROJECT 2:** Learn which style developers accept: short or detailed, imperative or descriptive.

We analyze the distribution of moods and lengths across real repository commits and show automated style transformation.

In [15]:
test_subjects = [
    "Adapt test_response_decode_text_using_autodetect for chardet 6.0",
    "Expose FunctionAuth in __all__",
    "docs/ssl: fix typo",
    "Fixed a syntax error in the file upload example",
    "Bump actions/setup-python from 5 to 6",
    "Update dependencies",
    "Documentation for SSL_CERT_FILE and SSL_CERT_DIR",
    "Updating deprecated docstring Client",
]

styles = analyze_styles(test_subjects)
print("=== Developer Commit Style Distribution ===")
print(json.dumps(styles, indent=2))

print("\n=== Dynamic Style Transformation ===")
raw_msg = "Fixes bug in connection handler, ensuring socket closes promptly"
print("Original:              ", raw_msg)
print("Imperative + Short:    ", format_style(raw_msg, mood="imperative", length="short"))
print("Descriptive + Short:   ", format_style(raw_msg, mood="descriptive", length="short"))

=== Developer Commit Style Distribution ===
{
  "total_messages": 8,
  "mood_distribution": {
    "imperative": {
      "count": 6,
      "percentage": 75.0
    },
    "descriptive": {
      "count": 2,
      "percentage": 25.0
    },
    "neutral": {
      "count": 0,
      "percentage": 0.0
    }
  },
  "length_distribution": {
    "short": {
      "count": 7,
      "percentage": 87.5
    },
    "detailed": {
      "count": 1,
      "percentage": 12.5
    }
  },
  "average_word_count": 4.75
}

=== Dynamic Style Transformation ===
Original:               Fixes bug in connection handler, ensuring socket closes promptly
Imperative + Short:     Fix bug in connection handler
Descriptive + Short:    Fixes bug in connection handler


## 8. Interactive CLI Tool (`commitgen`)

The CLI allows developers to:
1. `commitgen commit`: Inspect staged git changes, generate a concise message, and prompt for confirmation.
2. `commitgen docstring`: Inspect Python functions and generate one-line docstrings with optional in-place writing.

In [16]:
from commitgen.application.cli import generate_commit_suggestion, generate_docstring_suggestion

diff = """diff --git a/config.py b/config.py
--- a/config.py
+++ b/config.py
@@ -1,2 +1,3 @@
+TIMEOUT_SECONDS = 30
"""

code = """
def compute_hash(data: bytes) -> str:
    # Return hex digest of SHA-256 hash
    return hashlib.sha256(data).hexdigest()
"""

print("CLI Commit Suggestion:  ", generate_commit_suggestion(diff, model="baseline", mood="imperative"))
print("CLI Docstring Suggestion:", generate_docstring_suggestion(code, model="baseline", mood="imperative"))

CLI Commit Suggestion:   Update config
CLI Docstring Suggestion: Return hex digest of SHA-256 hash
